# Web scraping: building a DataFrame from a website

We collect company data (name, rating, reviews, type, headquarters, age, employees)
from the AmbitionBox list of companies, using `requests` and `BeautifulSoup`.

**Note (October 2026):** the live site no longer answers scripts, and it has a new layout.
So every step tries the live site first and falls back to real copies of the pages,
saved in `data/` from the Wayback Machine (August 2022, the layout this code was written for).

In [ ]:
# Libraries: pandas for tables, requests to download pages,
# BeautifulSoup to read HTML, numpy for NaN, time for pauses
import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

## 1. Downloading a page with requests

In [ ]:
# Page 1 of the list. Changing page=1 to page=2 gives the next 30 companies
URL = "https://www.ambitionbox.com/list-of-companies?page={}"

# A plain request: no browser identity. In 2021 this got status 403 (Forbidden);
# in 2026 the server does not answer at all, so we stop waiting after 10 seconds
try:
    r = requests.get(URL.format(1), timeout=10)
    print("status code:", r.status_code)
except requests.RequestException as e:
    print("no answer:", type(e).__name__)

In [ ]:
# The fix: send a User-Agent header, so the request looks like it comes from a browser
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/120.0 Safari/537.36"}


@lru_cache  # remember each page, so it is requested only once
def get_page(n):
    """HTML of list page n: live if the site answers with the old layout, else the saved copy."""
    try:
        r = requests.get(URL.format(n), headers=HEADERS, timeout=10)
        if r.status_code == 200 and "company-content-wrapper" in r.text:
            print(f"page {n}: live")
            return r.text
        print(f"page {n}: live status {r.status_code}, using saved copy")
    except requests.RequestException as e:
        print(f"page {n}: no answer ({type(e).__name__}), using saved copy")
    return Path(f"data/ambitionbox_2022_page{n}.html").read_text(encoding="utf-8")


webpage = get_page(1)
# webpage is one long string of HTML: the same text as "View Page Source" in a browser
print(len(webpage), "characters")
print(webpage[:300])

## 2. Parsing the HTML with BeautifulSoup

In [ ]:
# Turn the text into a tree of tags we can search.
# "html.parser" is built into Python; "lxml" is faster but must be installed
soup = BeautifulSoup(webpage, "html.parser")

# prettify() re-indents the HTML so nesting is easy to see (first 1,500 characters)
print(soup.prettify()[:1500])

## 3. find_all and .text

In [ ]:
# find_all returns a list of every matching tag. The page has one h1 tag
soup.find_all("h1")

In [ ]:
# [0] picks the first tag; .text gives the text inside it; .strip() removes spaces and newlines
soup.find_all("h1")[0].text.strip()

In [ ]:
# Every company name sits in an h2 tag
print(len(soup.find_all("h2")), "h2 tags")
print(repr(soup.find_all("h2")[0].text))   # raw text: full of tabs and newlines
for tag in soup.find_all("h2")[:10]:
    print(tag.text.strip())                 # cleaned

## 4. Selecting tags by class

In [ ]:
# The p tag is used for many things on the page, not only ratings
print(len(soup.find_all("p")), "p tags")
for tag in soup.find_all("p")[:6]:
    print(tag.text.strip())

In [ ]:
# Only p tags whose class is "rating". class_ has an underscore: "class" is a Python keyword
ratings = soup.find_all("p", class_="rating")
print(len(ratings), "ratings:", [t.text.strip() for t in ratings[:5]])

# Review counts are a tags with class "review-count"
reviews = soup.find_all("a", class_="review-count")
print(len(reviews), "review counts:", [t.text.strip() for t in reviews[:5]])

## 5. The problem: fields that do not line up

In [ ]:
# Type, headquarters, age and employees all share the class "infoEntity".
# Page 1: 30 companies x 4 = 120. Page 2 has only 117: some companies miss a field,
# and a flat list cannot tell us which ones
for n in [1, 2]:
    s = BeautifulSoup(get_page(n), "html.parser")
    print(f"page {n}:", len(s.find_all("p", class_="infoEntity")), "infoEntity tags")

## 6. Working container by container

In [ ]:
# Each company's details sit inside one div with class "company-content-wrapper"
company = soup.find_all("div", class_="company-content-wrapper")
len(company)

In [ ]:
# company[0] is the whole TCS box. Searching inside it finds only TCS's tags.
# find returns the first match (one tag); find_all returns a list
first = company[0]
print(first.find("h2").text.strip())
print(first.find("p", class_="rating").text.strip())
print([t.text.strip() for t in first.find_all("p", class_="infoEntity")])

## 7. Building the DataFrame for one page

In [ ]:
# One empty list per column
name, rating, reviews, ctype, hq, how_old, no_of_employee = [], [], [], [], [], [], []

# Visit each company box and append one value to each list
for i in company:
    info = i.find_all("p", class_="infoEntity")
    name.append(i.find("h2").text.strip())
    rating.append(i.find("p", class_="rating").text.strip())
    reviews.append(i.find("a", class_="review-count").text.strip())
    ctype.append(info[0].text.strip())
    hq.append(info[1].text.strip())
    how_old.append(info[2].text.strip())
    no_of_employee.append(info[3].text.strip())

# A dictionary of lists becomes a DataFrame: each key is a column
df = pd.DataFrame({"name": name, "rating": rating, "reviews": reviews,
                   "company_type": ctype, "head_quarters": hq,
                   "company_age": how_old, "no_of_employees": no_of_employee})
print(df.shape)
df.head()

In [ ]:
# Long names are cut short in the text ("Mahindra & Mahin...").
# The full name is in the h2 tag's title attribute; tag["title"] reads an attribute
for i in company:
    h2 = i.find("h2")
    if h2.text.strip().endswith("..."):
        print(h2.text.strip(), "->", h2["title"])

## 8. Scraping every page

In [ ]:
def text_or_nan(tag):
    """Clean text of a tag, or NaN when the tag was not found (find returned None)."""
    return tag.text.strip() if tag is not None else np.nan


def scrape_page(html):
    """One DataFrame row per company box on one list page."""
    rows = []
    for i in BeautifulSoup(html, "html.parser").find_all("div", class_="company-content-wrapper"):
        # the details as text, padded with NaN up to 4 when some are missing
        info = [p.text.strip() for p in i.find_all("p", class_="infoEntity")]
        info += [np.nan] * (4 - len(info))
        rows.append({"name": i.find("h2")["title"].strip(),
                     "rating": text_or_nan(i.find("p", class_="rating")),
                     "reviews": text_or_nan(i.find("a", class_="review-count")),
                     "company_type": info[0],
                     "head_quarters": info[1],
                     "company_age": info[2],
                     "no_of_employees": info[3]})
    return pd.DataFrame(rows)


# The real site had about 333 pages; we have saved copies of pages 1 and 2
pages = []
for j in range(1, 3):
    pages.append(scrape_page(get_page(j)))
    time.sleep(2)   # pause between requests: be polite to the server

# pd.concat stacks the DataFrames; ignore_index renumbers the rows 0, 1, 2, ...
final = pd.concat(pages, ignore_index=True)
print(final.shape)
final.sample(5, random_state=0)

In [ ]:
# Missing fields became NaN instead of crashing the loop
final.isna().sum()

## 9. A silent mistake: positions shift

Infosys BPM has no company type, so its three infoEntity tags are
headquarters, age and employees. Taking them by position puts each value one column too far left.

In [ ]:
final[final["name"].isin(["Infosys BPM", "HCL Group"])]

In [ ]:
# Safer: each infoEntity has an icon (an i tag) that says what it is
ICONS = {"icon-domain": "company_type", "icon-pin-drop": "head_quarters",
         "icon-access-time": "company_age", "icon-supervisor-account": "no_of_employees"}


def info_by_icon(box):
    """Map each infoEntity to its column by its icon, not by its position."""
    out = dict.fromkeys(ICONS.values(), np.nan)
    for p in box.find_all("p", class_="infoEntity"):
        out[ICONS[p.find("i")["class"][0]]] = p.text.strip()
    return out


page2 = BeautifulSoup(get_page(2), "html.parser").find_all("div", class_="company-content-wrapper")
infosys_bpm = [b for b in page2 if b.find("h2").text.strip() == "Infosys BPM"][0]
info_by_icon(infosys_bpm)

## 10. The same code on today's page

In [ ]:
# The site was redesigned. On a saved copy from August 2026, the old class names find nothing
new = BeautifulSoup(Path("data/ambitionbox_2026_page1.html").read_text(encoding="utf-8"),
                    "html.parser")
print(len(new.find_all("div", class_="company-content-wrapper")), "old-style boxes")
print(len(new.find_all("div", class_="companyCardWrapper")), "new-style boxes")

In [ ]:
# Same method, new class names (found with Inspect in the browser).
# Company type, age and employees are no longer on this page
rows = []
for box in new.find_all("div", class_="companyCardWrapper"):
    rows.append({"name": box.find("h2").text.strip(),
                 "rating": box.find("div", class_="rating_text").text.strip(),
                 "reviews": box.find("span", class_="companyCardWrapper__companyRatingCount").text.strip(),
                 "industry_and_location": box.find("span", class_="companyCardWrapper__interLinking").text.strip()})
pd.DataFrame(rows).head()